# 10 — Concatenación Features Red + Físicos
Join por timestamp exacto entre features de red y features físicas.
Resultado: dataset combinado listo para entrenar modelo conjunto.
Inner join — solo los 706.319 registros con match en ambos datasets.

In [0]:
from pyspark.sql import functions as F

df_red = spark.read.format("delta") \
    .load("/Volumes/workspace/default/network_data/features_fe/") \
    .select("window_start", "label")

df_fis = spark.read.format("delta") \
    .load("/Volumes/workspace/default/phisical_measures/features_fisicos/") \
    .select("timestamp_dt", "label")

# Rangos temporales
print("=== Red ===")
display(df_red.agg(
    F.min("window_start").alias("inicio"),
    F.max("window_start").alias("fin"),
    F.count("*").alias("total")
))

print("=== Fisicos ===")
display(df_fis.agg(
    F.min("timestamp_dt").alias("inicio"),
    F.max("timestamp_dt").alias("fin"),
    F.count("*").alias("total")
))

# Cuántos timestamps coinciden exactamente
df_join = df_red.join(
    df_fis,
    df_red["window_start"] == df_fis["timestamp_dt"],
    how="inner"
)
print(f"Registros con match exacto: {df_join.count():,}")

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np

DELTA_RED_PATH     = "/Volumes/workspace/default/network_data/features_fe/"
DELTA_FISICOS_PATH = "/Volumes/workspace/default/phisical_measures/features_fisicos/"
DELTA_COMBINED     = "/Volumes/workspace/default/network_data/features_combined/"

# Cargar ambos datasets
df_red = spark.read.format("delta").load(DELTA_RED_PATH)
df_fis = spark.read.format("delta").load(DELTA_FISICOS_PATH)

print(f"Red     : {df_red.count():,} registros, {len(df_red.columns)} columnas")
print(f"Fisicos : {df_fis.count():,} registros, {len(df_fis.columns)} columnas")

## 1 — Preparar columnas antes del join

In [0]:
# Renombrar label de físicos para evitar ambigüedad tras el join
# Solo nos quedamos con el label de red — es el que hemos validado
# durante todo el pipeline (corrección con datos físicos en NB 05)
df_fis = df_fis.drop("label") \
               .withColumnRenamed("timestamp_dt", "ts_fisicos")

# Verificar que no hay columnas duplicadas entre los dos datasets
cols_red = set(df_red.columns) - {"window_start", "window_end",
                                   "window_id", "session_id", "label"}
cols_fis = set(df_fis.columns) - {"ts_fisicos"}
duplicadas = cols_red.intersection(cols_fis)

print(f"Columnas duplicadas entre datasets: {duplicadas}")
# Si hay duplicadas las renombramos con prefijo
for col in duplicadas:
    df_fis = df_fis.withColumnRenamed(col, f"fis_{col}")

print(f"Columnas red     : {len(df_red.columns)}")
print(f"Columnas fisicos : {len(df_fis.columns)}")

## 2 — Inner join por timestamp exacto

In [0]:
# Inner join por segundo exacto
# Solo los registros que tienen lectura en ambos datasets
df_combined = df_red.join(
    df_fis,
    df_red["window_start"] == df_fis["ts_fisicos"],
    how="inner"
).drop("ts_fisicos")

total_combined = df_combined.count()
print(f"Registros combinados : {total_combined:,}")
print(f"Columnas totales     : {len(df_combined.columns)}")

# Verificar distribución de labels
display(
    df_combined.groupBy("label")
    .count()
    .withColumn("pct", F.round(F.col("count") / total_combined * 100, 2))
    .orderBy("label")
)

## 3 — Verificar integridad del dataset combinado

In [0]:
# Comprobar que no hay NULLs en ninguna columna
# Un NULL indicaría un problema en el join o en los datos de origen
exclude_check = ["window_end", "window_id", "session_id"]
check_cols = [c for c in df_combined.columns if c not in exclude_check]

null_counts = df_combined.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in check_cols
]).toPandas().T

null_counts.columns = ["nulls"]
null_counts = null_counts[null_counts["nulls"] > 0]

if len(null_counts) == 0:
    print("OK — No hay NULLs en ninguna columna")
else:
    print(f"ATENCION — Columnas con NULLs:")
    print(null_counts.to_string())

# Rango temporal del dataset combinado
display(
    df_combined.agg(
        F.min("window_start").alias("inicio"),
        F.max("window_start").alias("fin"),
        F.count("*").alias("total")
    )
)

## 4 — Guardar dataset combinado

In [0]:
df_combined.repartition(64) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(DELTA_COMBINED)

print(f"Guardado en: {DELTA_COMBINED}")

# Verificación final
df_check = spark.read.format("delta").load(DELTA_COMBINED)
total_check = df_check.count()
print(f"Filas  : {total_check:,}")
print(f"Cols   : {len(df_check.columns)}")
print(f"Schema:")
df_check.printSchema()

In [0]:
print("=" * 60)
print("RESUMEN — CONCATENACION RED + FISICOS")
print("=" * 60)
print(f"Registros red           : 806.400")
print(f"Registros fisicos       : 846.719")
print(f"Match exacto (inner)    : {total_combined:,}")
print(f"Perdidos de red         : {806400 - total_combined:,}  (sin match en fisicos)")
print(f"Perdidos de fisicos     : {846719 - total_combined:,}  (sin match en red)")
print("-" * 60)
dist = df_check.groupBy("label").count().orderBy("label").toPandas()
for _, row in dist.iterrows():
    pct = row["count"] / total_combined * 100
    nombre = "Normal (0)" if row["label"] == 0 else "Ataque (1)"
    print(f"  {nombre}  : {row['count']:,}  ({pct:.2f}%)")
print("-" * 60)
print(f"Columnas red            : {len(df_red.columns)}")
print(f"Columnas fisicos        : {len(df_fis.columns) + 1}")  # +1 por label
print(f"Columnas combinadas     : {len(df_check.columns)}")
print(f"Delta guardado          : {DELTA_COMBINED}")
print("=" * 60)